# Week 1: Python & NumPy Mastery
## Notebook 13: Moore-Penrose Pseudoinverse & Minimum Norm Solutions

### 1. Conceptual Intuition & Theoretical Background
When solving linear system $A x = b$:
- If $A$ is square and non-singular: $x = A^{-1} b$.
- If $A$ is **underdetermined** (more unknowns than equations) or **overdetermined** (more equations than unknowns), $A^{-1}$ does not exist.

The **Moore-Penrose Pseudoinverse ($A^+$)** provides the unique optimal solution satisfying four Penrose conditions:
1. $A A^+ A = A$
2. $A^+ A A^+ = A^+$
3. $(A A^+)^T = A A^+$
4. $(A^+ A)^T = A^+ A$

Using SVD ($A = U \Sigma V^T$), the pseudoinverse is computed by inverting non-zero singular values:
$$A^+ = V \Sigma^+ U^T, \quad \Sigma^+_{ii} = \begin{cases} \frac{1}{\sigma_i} & \text{if } \sigma_i > \epsilon \\ 0 & \text{if } \sigma_i \le \epsilon \end{cases}$$

For underdetermined systems, $x = A^+ b$ selects the solution with the **minimum Euclidean $L_2$ norm** ($\|x\|_2$).

---

### 2. Scratch Python Implementation: Moore-Penrose Solver

In [ ]:
import numpy as np

def moore_penrose_pseudoinverse(A, rtol=1e-12):
    """Computes Moore-Penrose pseudoinverse A^+ using SVD."""
    U, s, Vt = np.linalg.svd(A, full_matrices=False)
    
    # Cutoff threshold for small singular values
    max_s = np.max(s)
    cutoff = rtol * max_s
    
    # Invert singular values above cutoff
    s_inv = np.zeros_like(s)
    mask = s > cutoff
    s_inv[mask] = 1.0 / s[mask]
    
    # A^+ = V @ Sigma^+ @ U^T
    A_plus = np.dot(Vt.T, s_inv[:, np.newaxis] * U.T)
    return A_plus

# 1. Underdetermined System: 3 Equations, 5 Unknowns (Infinite solutions exist)
np.random.seed(42)
A_under = np.array([
    [1.0, 2.0, 0.0, 1.0, 1.0],
    [0.0, 1.0, 3.0, 2.0, 0.0],
    [2.0, 0.0, 1.0, 0.0, 2.0]
])
b_target = np.array([5.0, 8.0, 6.0])

A_pinv = moore_penrose_pseudoinverse(A_under)
x_min_norm = np.dot(A_pinv, b_target)

# Check equation satisfiability: A @ x == b
reconstructed_b = np.dot(A_under, x_min_norm)
residual_norm = np.linalg.norm(reconstructed_b - b_target)

# Compare with alternative random valid solution
null_vector = np.array([2.0, -1.0, 0.5, -0.25, -1.0])
null_proj = null_vector - np.dot(A_pinv @ A_under, null_vector)
x_alt = x_min_norm + null_proj

print("--- Moore-Penrose Pseudoinverse Solution ---")
print(f"Target vector b:             {b_target}")
print(f"Reconstructed vector A @ x:  {np.round(reconstructed_b, 4)}")
print(f"Fit Residual Error:          {residual_norm:.1e}")
print(f"L2 Norm of Pseudoinverse x:  {np.linalg.norm(x_min_norm):.4f}")
print(f"L2 Norm of Alternative sol:  {np.linalg.norm(x_alt):.4f}")
print("Verified: Pseudoinverse produces the unique minimum-energy/norm solution!")

### 3. Step-by-Step Code Walkthrough
1. **Singular Value Thresholding:** Zeroes out singular values below $\epsilon$ to avoid numerical explosion.
2. **Penrose Properties:** Evaluates exact least squares fit for rank-deficient matrices.
3. **Minimum Norm Property:** Proves $\|x_{\text{pinv}}\|_2 < \|x_{\text{alt}}\|_2$.

---

### 4. Industry Applications & Trade-offs
- **Robotics Inverse Kinematics:** Computing joint angle velocities from end-effector spatial coordinates when robot arms have redundant degrees of freedom.
- **Trade-off:** High condition numbers (ratio $\sigma_{\max} / \sigma_{\min}$) make naive pseudoinverse inversion sensitive to sensor measurement noise without ridge damping.

---

### 5. Top Interview Questions & Answers
**Q1: How does the pseudoinverse solution differ between overdetermined and underdetermined systems?**
> **Answer:** In overdetermined systems ($m > n$), $x = A^+ b$ minimizes the residual sum of squares $\|A x - b\|_2$. In underdetermined systems ($m < n$), it selects the unique exact solution that minimizes the parameter norm $\|x\|_2$.

**Q2: What is the condition number of a matrix in relation to the pseudoinverse?**
> **Answer:** $\kappa(A) = \frac{\sigma_{\max}}{\sigma_{\min}}$. A very large condition number means the matrix is nearly singular, meaning small perturbations in $b$ cause massive swings in $x$.